# Testing file 
### where we evaluate Zhang's models using the test set

## Preliminaries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.data import Dataset


from util.load_data import load_data
from util.evaluation import *
from models.zhang.models import FairLogisticRegression
from models.zhang.learning import train_loop as zhang_train

/Users/lffpl/Projects/falsb/env/falsb/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
batch_size = 64
epochs = 100
lr = 0.001

In [3]:
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'balanced-stroke-age'

In [5]:
x, y, a = load_data(data_name)
raw_data = (x, y, a)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
adim = a.shape[1]
zdim = 8

In [7]:
print(xdim, ydim, adim, zdim)

17 1 1 8


In [8]:
print(len(x))

9398


## Result file

In [9]:
header = "model_name", "cv_seed", "clas_acc", "dp", "deqodds", "deqopp", "trade_dp", "trade_deqodds", "trade_deqopp", "TN_a0", "FP_a0", "FN_a0", "TP_a0", "TN_a1", "FP_a1", "FN_a1", "TP_a1"
results = []

## Testing loop
#### Each model is evalueted 5 times
#### In the end of each iteration we save the result

### Zhang for DP

In [10]:
fairdef = 'DemPar'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)

    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4DP', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc


2026-01-05 17:26:23.110070: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 1 | 0.6449070572853088 | 0.7627347707748413 | 0.5416666666666666 | 0.39047181372549017


2026-01-05 17:26:25.378745: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 2 | 0.5990229845046997 | 0.7588627338409424 | 0.6798406862745098 | 0.39047181372549017
> 3 | 0.5682822465896606 | 0.7582939863204956 | 0.7377450980392157 | 0.39047181372549017


...

> 64 | 0.28455501794815063 | 0.6258549690246582 | 0.8912377450980392 | 0.4878982843137255
> 65 | 0.28440481424331665 | 0.6247893571853638 | 0.8920036764705882 | 0.4898897058823529
> 66 | 0.28316283226013184 | 0.6295968294143677 | 0.8909313725490197 | 0.4883578431372549
> 67 | 0.2841299772262573 | 0.6228151321411133 | 0.8910845588235294 | 0.4889705882352941
> 68 | 0.2842494249343872 | 0.6216821074485779 | 0.891390931372549 | 0.4944852941176471
> 69 | 0.2841147184371948 | 0.6211492419242859 | 0.891390931372549 | 0.5016850490196079
> 70 | 0.2841236889362335 | 0.6199952960014343 | 0.8920036764705882 | 0.5027573529411765
> 71 | 0.2841589152812958 | 0.618904709815979 | 0.8900122549019608 | 0.5016850490196079
> 72 | 0.28419822454452515 | 0.6208891868591309 | 0.890625 | 0.5032169117647058
> 73 | 0.2839509844779968 | 0.617375373840332 | 0.890625 | 0.5055147058823529
> 74 | 0.28399497270584106 | 0.6162128448486328 | 0.890625 | 0.5050551470588235
> 75 | 0.2838103771209717 | 0.6163020730018616 | 0

2026-01-05 17:31:17.546933: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 27 | 0.2746744751930237 | 0.7750357389450073 | 0.8893995098039216 | 0.39169730392156865
> 28 | 0.2729266285896301 | 0.7691599130630493 | 0.8910845588235294 | 0.3915441176470588
> 29 | 0.27159231901168823 | 0.7632337212562561 | 0.890625 | 0.39139093137254904
> 30 | 0.269262433052063 | 0.7570757865905762 | 0.8907781862745098 | 0.3915441176470588
> 31 | 0.26815474033355713 | 0.7512372732162476 | 0.8915441176470589 | 0.3996629901960784
> 32 | 0.26724371314048767 | 0.7455822229385376 | 0.8929227941176471 | 0.4022671568627451
> 33 | 0.2665790319442749 | 0.7399686574935913 | 0.8938419117647058 | 0.40732230392156865
> 34 | 0.2672145962715149 | 0.7346906065940857 | 0.8950674019607843 | 0.41452205882352944
> 35 | 0.26683199405670166 | 0.7292811274528503 | 0.8953737745098039 | 0.4143688725490196
> 36 | 0.26672205328941345 | 0.7240386009216309 | 0.8956801470588235 | 0.41421568627450983
> 37 | 0.26664575934410095 | 0.7189631462097168 | 0.8947610294117647 | 0.4224877450980392
> 38 | 0.266705095767

2026-01-05 17:36:16.161988: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 54 | 0.40856409072875977 | 0.7104595899581909 | 0.8949142156862745 | 0.46859681372549017
> 55 | 0.4093264043331146 | 0.7103781700134277 | 0.8949142156862745 | 0.47028186274509803
> 56 | 0.41006696224212646 | 0.7102994918823242 | 0.8949142156862745 | 0.4696691176470588
> 57 | 0.4107493460178375 | 0.7102963924407959 | 0.8952205882352942 | 0.46798406862745096
> 58 | 0.411176472902298 | 0.710307240486145 | 0.8950674019607843 | 0.46675857843137253
> 59 | 0.4141187071800232 | 0.7094616293907166 | 0.8941482843137255 | 0.46875
> 60 | 0.41445156931877136 | 0.7095740437507629 | 0.8949142156862745 | 0.47472426470588236
> 61 | 0.4174138009548187 | 0.7066495418548584 | 0.8949142156862745 | 0.47349877450980393
> 62 | 0.4179176688194275 | 0.7084852457046509 | 0.8941482843137255 | 0.47579656862745096
> 63 | 0.41832518577575684 | 0.7084014415740967 | 0.8949142156862745 | 0.48590686274509803
> 64 | 0.4213596284389496 | 0.7074453830718994 | 0.8936887254901961 | 0.48575367647058826
> 65 | 0.421605855226

### Zhang for Eq Odds

In [11]:
fairdef = 'EqOdds'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.6393399834632874 | 0.7578697204589844 | 0.5381433823529411 | 0.39047181372549017
> 2 | 0.6053357124328613 | 0.752539336681366 | 0.6683517156862745 | 0.39047181372549017
> 3 | 0.5702157616615295 | 0.7470648884773254 | 0.7280943627450981 | 0.39047181372549017
> 4 | 0.5421854257583618 | 0.7444489002227783 | 0.7709865196078431 | 0.39047181372549017
> 5 | 0.51251220703125 | 0.7407332062721252 | 0.7999387254901961 | 0.39016544117647056
> 6 | 0.4902164340019226 | 0.7383158206939697 | 0.8293504901960784 | 0.3883272058823529


2026-01-05 17:46:13.855441: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 7 | 0.4705810844898224 | 0.7354308366775513 | 0.842984068627451 | 0.38235294117647056
> 8 | 0.45315760374069214 | 0.7323395013809204 | 0.8498774509803921 | 0.3756127450980392
> 9 | 0.4357476830482483 | 0.7267521619796753 | 0.8566176470588235 | 0.36611519607843135
> 10 | 0.42455101013183594 | 0.7254283428192139 | 0.8595281862745098 | 0.36106004901960786
> 11 | 0.4101574122905731 | 0.717644214630127 | 0.8616727941176471 | 0.3556985294117647
> 12 | 0.398581326007843 | 0.7114208936691284 | 0.8627450980392157 | 0.3486519607843137
> 13 | 0.3879638910293579 | 0.7109783887863159 | 0.8688725490196079 | 0.34650735294117646
> 14 | 0.37912267446517944 | 0.710797905921936 | 0.875765931372549 | 0.3431372549019608
> 15 | 0.37139594554901123 | 0.7114781141281128 | 0.8788296568627451 | 0.34068627450980393
> 16 | 0.3641151785850525 | 0.7109076976776123 | 0.8794424019607843 | 0.34221813725490197
> 17 | 0.35727912187576294 | 0.711028516292572 | 0.8821997549019608 | 0.34359681372549017
> 18 | 0.349741876

### Zhang for Eq Opp

In [12]:
fairdef = 'EqOpp'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOpp', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.6328434348106384 | 0.33541446924209595 | 0.5419730392156863 | 0.39047181372549017
> 2 | 0.600817084312439 | 0.3350779116153717 | 0.6778492647058824 | 0.39047181372549017
> 3 | 0.5673673152923584 | 0.33491814136505127 | 0.7378982843137255 | 0.39047181372549017
> 4 | 0.5362504124641418 | 0.33408448100090027 | 0.7821691176470589 | 0.39047181372549017
> 5 | 0.5089687705039978 | 0.3341618776321411 | 0.8095894607843137 | 0.39047181372549017
> 6 | 0.4849364459514618 | 0.33555135130882263 | 0.8301164215686274 | 0.39047181372549017
> 7 | 0.46584171056747437 | 0.33483701944351196 | 0.8443627450980392 | 0.39047181372549017
> 8 | 0.4470440149307251 | 0.3365771770477295 | 0.852328431372549 | 0.39047181372549017
> 9 | 0.43151235580444336 | 0.3368915021419525 | 0.8578431372549019 | 0.39047181372549017
> 10 | 0.41441023349761963 | 0.3364271819591522 | 0.8604473039215687 | 0.39047181372549017
> 11 | 0.4042772054672241 | 0.33799365162849426 |

2026-01-05 18:10:22.659456: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 14 | 0.37258702516555786 | 0.33815500140190125 | 0.874234068627451 | 0.39047181372549017
> 15 | 0.3636704683303833 | 0.3385660648345947 | 0.8754595588235294 | 0.39047181372549017
> 16 | 0.3553617596626282 | 0.3389521837234497 | 0.8780637254901961 | 0.39047181372549017
> 17 | 0.3477144241333008 | 0.3393248915672302 | 0.8809742647058824 | 0.39047181372549017
> 18 | 0.3379268944263458 | 0.33946311473846436 | 0.8871017156862745 | 0.39047181372549017
> 19 | 0.331523060798645 | 0.33976107835769653 | 0.8864889705882353 | 0.39047181372549017
> 20 | 0.3256574273109436 | 0.3400637209415436 | 0.8883272058823529 | 0.39047181372549017
> 21 | 0.3190861940383911 | 0.340389609336853 | 0.8921568627450981 | 0.39047181372549017
> 22 | 0.313440203666687 | 0.34066370129585266 | 0.8927696078431373 | 0.39047181372549017
> 23 | 0.30849623680114746 | 0.34091976284980774 | 0.8947610294117647 | 0.39047181372549017
> 24 | 0.30400410294532776 | 0.34118425846099854 | 0.8962928921568627 | 0.39047181372549017
> 25 

## Saving into DF then CSV

In [13]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,dp,deqodds,deqopp,trade_dp,trade_deqodds,trade_deqopp,TN_a0,FP_a0,FN_a0,TP_a0,TN_a1,FP_a1,FN_a1,TP_a1
0,Zhang4DP,13,0.892045,0.520800,0.868826,0.918123,0.657648,0.880283,0.904896,265.0,67.0,131.0,882.0,1055.0,23.0,83.0,310.0
1,Zhang4DP,29,0.890270,0.487982,0.819225,0.892197,0.630416,0.853271,0.891232,246.0,91.0,115.0,926.0,1027.0,17.0,86.0,308.0
2,Zhang4DP,42,0.877131,0.448836,0.760886,0.822347,0.593812,0.814883,0.848856,204.0,102.0,97.0,909.0,1063.0,36.0,111.0,294.0
3,Zhang4DP,55,0.872869,0.423445,0.714156,0.856451,0.570251,0.785577,0.864582,188.0,161.0,77.0,924.0,1049.0,36.0,84.0,297.0
4,Zhang4DP,73,0.877486,0.471714,0.784921,0.875217,0.613582,0.828626,0.876350,214.0,116.0,95.0,937.0,1013.0,49.0,85.0,307.0
5,Zhang4EqOdds,13,0.889205,0.516403,0.859011,0.916566,0.653366,0.873847,0.902678,259.0,73.0,132.0,881.0,1055.0,23.0,84.0,309.0
6,Zhang4EqOdds,29,0.884943,0.502828,0.831451,0.910994,0.641279,0.857363,0.897780,245.0,92.0,124.0,917.0,1018.0,26.0,82.0,312.0
7,Zhang4EqOdds,42,0.873580,0.460739,0.772123,0.832288,0.603292,0.819724,0.852434,207.0,99.0,107.0,899.0,1060.0,39.0,111.0,294.0
8,Zhang4EqOdds,55,0.872514,0.419800,0.709061,0.848204,0.566861,0.782341,0.860188,187.0,162.0,74.0,927.0,1048.0,37.0,86.0,295.0
9,Zhang4EqOdds,73,0.875710,0.471259,0.782556,0.865369,0.612762,0.826517,0.870509,216.0,114.0,98.0,934.0,1014.0,48.0,90.0,302.0


In [14]:
result_df.to_csv(f'{data_name}-result/zhang-{epochs}.csv')